**Conversational Memory for LangChain**

Conversational memory allows our chatbots and agents to remember previous interactions within a conversation. Without conversational memory, our chatbots would only ever be able to respond to the last message they received, essentially forgetting all previous messages with each new message.

Naturally, conversations require our chatbots to be able to respond over multiple interactions and refer to previous messages to understand the context of the conversation.

**Importing the libraries and the model**

In [1]:
import langchain_community
import langchain_core
import langchain_ollama
import langchain
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

c:\Users\PatelDharmikkumar\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**LangChain's Memory Types**

LangChain versions 0.0.x consisted of various conversational memory types. Most of these are due for deprecation but still hold value in understanding the different approaches that we can take to building conversational memory.

Throughout the notebook we will be referring to these older memory types and then rewriting them using the recommended RunnableWithMessageHistory class. We will learn about:

ConversationBufferMemory: the simplest and most intuitive form of conversational memory, keeping track of a conversation without any additional bells and whistles.
ConversationBufferWindowMemory: similar to ConversationBufferMemory, but only keeps track of the last k messages.
ConversationSummaryMemory: rather than keeping track of the entire conversation, this memory type keeps track of a summary of the conversation.
ConversationSummaryBufferMemory: merges the ConversationSummaryMemory and ConversationTokenBufferMemory types.
We'll work through each of these memory types in turn, and rewrite each one using the RunnableWithMessageHistory class.

**1. ConversationBufferMemory**

ConversationBufferMemory is the simplest form of conversational memory, it is literally just a place that we store messages, and then use to feed messages into our LLM.

Let's start with LangChain's original ConversationBufferMemory object, we are setting return_messages=True to return the messages as a list of ChatMessage objects — unless using a non-chat model we would always set this to True as without it the messages are passed as a direct string which can lead to unexpected behavior from chat LLMs.

In [2]:
from langchain_community.chat_message_histories import ChatMessageHistory

memory = ChatMessageHistory()
memory.add_user_message("Hi, I am learning LangChain!")
memory.add_ai_message("That's awesome!")

print(memory.messages)

[HumanMessage(content='Hi, I am learning LangChain!', additional_kwargs={}, response_metadata={}), AIMessage(content="That's awesome!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


There are several ways that we can add messages to our memory, using the save_context method we can add a user query (via the input key) and the AI's response (via the output key). So, to create the following conversation:

User: Hi, my name is James

AI: Hey James, what's up? I'm an AI model called Zeta.

User: I'm researching the different types of conversational memory.

AI: That's interesting, what are some examples?

User: I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.

AI: That's interesting, what's the difference?

User: Buffer memory just stores the entire conversation, right?

AI: That makes sense, what about ConversationBufferWindowMemory?

User: Buffer window memory stores the last k messages, dropping the rest.

AI: Very cool!

We do:

In [3]:
from langchain_community.chat_message_histories import ChatMessageHistory

# Initialize the modern chat message history tracker
memory = ChatMessageHistory(return_messages=True)

# Turn 1
memory.add_user_message("Hi, my name is James")
memory.add_ai_message("Hey James, what's up? I'm an AI model called Zeta.")

# Turn 2
memory.add_user_message("I'm researching the different types of conversational memory.")
memory.add_ai_message("That's interesting, what are some examples?")

# Turn 3
memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.add_ai_message("That's interesting, what's the difference?")

# Turn 4
memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")

# Turn 5
memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.add_ai_message("Very cool!")

# How to view your message history now
print(memory.messages)

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey James, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}), AIMessage(content='That makes sense, what about Co

Before using the memory, we need to load in any variables for that memory type — in this case, there are none, so we just pass an empty dictionary:

In [4]:
# Returns a clean Python list directly: [HumanMessage(...), AIMessage(...)]
print(memory.messages)

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey James, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}), AIMessage(content='That makes sense, what about Co

In [5]:
memory

InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey James, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}), AIMessage(cont

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Building the pipeline (Prompt -> LLM)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])
base_chain = prompt | llm

# 2. Wrapping it so the same in-memory history is used for this demo session.
chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history=lambda session_id: memory,
    input_messages_key="input",
    history_messages_key="history"
)

C:\Users\PatelDharmikkumar\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
# Quick sanity check: the existing memory should still know James.
print(memory.messages[:2])

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey James, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [9]:
output = chain_with_history.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "demo-session"}}
)
print(output.content)

Your name is James.


**ConversationBufferMemory with RunnableWithMessageHistory**

As mentioned, the ConversationBufferMemory type is due for deprecation. Instead, we can use the RunnableWithMessageHistory class to implement the same functionality.

When implementing RunnableWithMessageHistory we will use LangChain Expression Language (LCEL) and for this we need to define our prompt template and LLM components. Our llm has already been defined, so now we just define a ChatPromptTemplate object.

In [10]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    ChatPromptTemplate
)

system_prompt = "You are a helpful assistant called Zeta."

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    MessagesPlaceholder(variable_name="history"),#This will allow us to use the history of the conversation as a part of the prompt, this will allow us to provide context to the model and also allow us to use the history of the conversation to provide examples to the model.
    HumanMessagePromptTemplate.from_template("{query}"),
])

We can link our prompt_template and our llm together to create a pipeline via LCEL.

In [11]:
pipeline = prompt_template | llm

Our RunnableWithMessageHistory requires our pipeline to be wrapped in a RunnableWithMessageHistory object. This object requires a few input parameters. One of those is get_session_history, which requires a function that returns a ChatMessageHistory object based on a session ID. We define this function ourselves:

In [12]:
from langchain_core.chat_history import InMemoryChatMessageHistory

buffer_chat_map = {}#This will be our in-memory chat history map, it will store the chat history for each session ID.
def get_buffer_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in buffer_chat_map:
        # if session ID doesn't exist, create a new chat history
        buffer_chat_map[session_id] = InMemoryChatMessageHistory()
    return buffer_chat_map[session_id]#Return the chat history for the given session ID.


We also need to tell our runnable which variable name to use for the chat history (ie history) and which to use for the user's query (ie query).

In [13]:
from langchain_core.runnables.history import RunnableWithMessageHistory

buffer_pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_buffer_chat_history,
    input_messages_key="query",
    history_messages_key="history"
)


C:\Users\PatelDharmikkumar\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
buffer_pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"session_id": "demo-session"}
)


AIMessage(content="Nice to meet you, James! I'm Zeta, your friendly assistant here to help with any questions or tasks you may have. How's your day going so far? Is there anything specific on your mind that you'd like some assistance with?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:31:31.5752768Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9673185800, 'load_duration': 283780700, 'prompt_eval_count': 30, 'prompt_eval_duration': 1201410600, 'eval_count': 51, 'eval_duration': 8046850500, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44b9-b5aa-7f31-8e80-0c6d90cfa36a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 51, 'total_tokens': 81})

In [15]:
buffer_pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={"session_id": "demo-session"}
)


AIMessage(content='Your name is James.', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:31:38.0980748Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3661870700, 'load_duration': 1717967400, 'prompt_eval_count': 96, 'prompt_eval_duration': 1098705000, 'eval_count': 6, 'eval_duration': 831244600, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44b9-e6a0-7150-9b67-6685549d30e9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 6, 'total_tokens': 102})

We have now recreated the ConversationBufferMemory type using the RunnableWithMessageHistory class. Let's continue onto other memory types and see how these can be implemented.

**2. ConversationBufferWindowMemory with RunnableWithMessageHistory**

The ConversationBufferWindowMemory type is similar to ConversationBufferMemory, but only keeps track of the last k messages. There are a few reasons why we would want to keep only the last k messages:

More messages mean more tokens are sent with each request, more tokens increases latency and cost.

LLMs tend to perform worse when given more tokens, making them more likely to deviate from instructions, hallucinate, or "forget" information provided to them. Conciseness is key to high performing LLMs.

If we keep all messages we will eventually hit the LLM's context window limit, by adding a window size k we can ensure we never hit this limit.

The buffer window solves many problems that we encounter with the standard buffer memory, while still being a very simple and intuitive form of conversational memory.

To implement this memory type using the RunnableWithMessageHistory class, we can use the same approach as before. We define our prompt_template and llm as before, and then wrap our pipeline in a RunnableWithMessageHistory object

For the window feature, we need to define a custom version of the InMemoryChatMessageHistory class that removes any messages beyond the last k messages.

In [16]:
from pydantic import BaseModel, Field
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
#making a class withbuddefwindowmemory that inherits from BaseChatMessageHistory and BaseModel, it will have a list of messages and an integer k that will determine how many messages to keep in the history.
#it'll also have methods to add messages to our class and also remove messages that are beyond the last k messages in the history so that we can keep the history manageable and not overloading the model with too much context.
class BufferWindowMessageHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)
    k: int = Field(default_factory=int)

    def __init__(self, k: int):
        super().__init__(k=k)
        print(f"Initializing BufferWindowMessageHistory with k={k}")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond
        the last `k` messages.
        """
        self.messages.extend(messages)
        self.messages = self.messages[-self.k:]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

In [17]:
from langchain_core.chat_history import InMemoryChatMessageHistory  #making a new chat map that will use our BufferWindowMessageHistory instead of the InMemoryChatMessageHistory, this will allow us to keep only the last k messages in the history and drop the rest.
window_chat_map = {}
#making a method to get the chat history for a given session ID,if the session ID deosn't exist,it'll create a new BufferWindowMessageHistory with the given k and store it in the chat map,then it'll return the chat history for the given session ID.
def get_window_chat_history(session_id: str, k: int = 4) -> InMemoryChatMessageHistory:
    print(f"get_window_chat_history called with session_id={session_id} and k={k}")
    if session_id not in window_chat_map:
        # if session ID doesn't exist, create a new chat history
        window_chat_map[session_id] = InMemoryChatMessageHistory()
    # remove anything beyond the last
    return window_chat_map[session_id]


In [18]:
from langchain_core.runnables import ConfigurableFieldSpec #configurable field spec will allow us to specify the configuration for our runnable, in this case we want to specify the session ID and the k value for our BufferWindowMessageHistory.
from langchain_core.runnables.history import RunnableWithMessageHistory


window_pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_window_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)


Now we invoke our runnable, this time passing a k parameter via the config parameter.

In [19]:
window_pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "demo", "k": 4}}
)


get_window_chat_history called with session_id=demo and k=4


AIMessage(content="Nice to meet you, James! I'm Zeta, your friendly assistant here to help with any questions or tasks you may have. How's your day going so far? Is there anything specific on your mind that you'd like some assistance with?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:31:57.2197278Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8819820800, 'load_duration': 548394500, 'prompt_eval_count': 30, 'prompt_eval_duration': 172645800, 'eval_count': 51, 'eval_duration': 7948046400, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44ba-1d2c-7bb2-8184-96fcd5481c84-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 51, 'total_tokens': 81})

In [20]:
window_chat_map["demo"].clear()  # clear the history

# manually insert history
window_chat_map["demo"].add_user_message("Hi, my name is James")
window_chat_map["demo"].add_ai_message("I'm an AI model called Zeta.")
window_chat_map["demo"].add_user_message("I'm researching the different types of conversational memory.")
window_chat_map["demo"].add_ai_message("That's interesting, what are some examples?")
window_chat_map["demo"].add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
window_chat_map["demo"].add_ai_message("That's interesting, what's the difference?")
window_chat_map["demo"].add_user_message("Buffer memory just stores the entire conversation, right?")
window_chat_map["demo"].add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
window_chat_map["demo"].add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
window_chat_map["demo"].add_ai_message("Very cool!")

window_chat_map["demo"].messages


[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, what about ConversationBuffe

In [21]:
window_pipeline_with_history.invoke(
    {"query": "what is my name again?"},
    config={"configurable": {"session_id": "demo", "k": 4}}
)


get_window_chat_history called with session_id=demo and k=4


AIMessage(content="Your name is James. I'm Zeta, your helpful assistant.", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:32:14.7052226Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11653617000, 'load_duration': 282668200, 'prompt_eval_count': 175, 'prompt_eval_duration': 8881868200, 'eval_count': 15, 'eval_duration': 2454785000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44ba-5667-7062-bb24-3fb92799efa7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 175, 'output_tokens': 15, 'total_tokens': 190})

That's it! We've rewritten our buffer window memory using the recommended RunnableWithMessageHistory class.

**3. ConversationSummaryMemory**

Next up we have ConversationSummaryMemory, this memory type keeps track of a summary of the conversation rather than the entire conversation. This is useful for long conversations where we don't need to keep track of the entire conversation, but we do want to keep some thread of the full conversation.

As before, we'll start with the original memory class before reimplementing it with the RunnableWithMessageHistory class.

In [23]:
from typing import Any

from pydantic import ConfigDict
from langchain_core.messages import SystemMessage


class ConversationSummaryMessageHistory(BaseChatMessageHistory, BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    llm: Any
    messages: list[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Summarize prior context plus new messages into one system message."""
        existing_summary = """"""
        if self.messages and isinstance(self.messages[0], SystemMessage):
            existing_summary = self.messages[0].content

        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensuring to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{messages}"
            )
        ])
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary,
                messages=[x.content for x in messages]
            )
        )
        self.messages = [SystemMessage(content=new_summary.content)]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []


In [24]:
summary_chat_map = {}
def get_summary_chat_history(session_id: str) -> ConversationSummaryMessageHistory:
    if session_id not in summary_chat_map:
        # if session ID doesn't exist, create a new chat history
        summary_chat_map[session_id] = ConversationSummaryMessageHistory(llm=llm)
    # return the chat history
    return summary_chat_map[session_id]


In [25]:
summary_pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_summary_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        )
    ]
)


C:\Users\PatelDharmikkumar\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Now we invoke our runnable, this time passing a llm parameter via the config parameter.

In [26]:
summary_pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "demo2"}}
)


AIMessage(content="Nice to meet you, James! I'm Zeta, your friendly assistant here to help with any questions or tasks you may have. How's your day going so far? Is there anything specific on your mind that you'd like some assistance with?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:34:32.2144582Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8545891700, 'load_duration': 584554200, 'prompt_eval_count': 30, 'prompt_eval_duration': 227099700, 'eval_count': 51, 'eval_duration': 7398391300, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44bc-7bb1-70e3-b7c3-71f52723cd97-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 51, 'total_tokens': 81})

Let's see what summary was generated:

In [27]:
summary_chat_map["demo2"].messages

[SystemMessage(content='Here is a new summary of the conversation:\n\nConversation Summary:\nJames and Zeta are having an introductory conversation, where they are exchanging greetings and getting to know each other. James has introduced himself as "Hi, my name is James", and Zeta has responded with a friendly greeting, offering assistance with any questions or tasks James may have.', additional_kwargs={}, response_metadata={})]

In [28]:
summary_pipeline_with_history.invoke(
    {"query": "I'm researching the different types of conversational memory."},
    config={"configurable": {"session_id": "demo2"}}
)

summary_chat_map["demo2"].messages


[SystemMessage(content='Here is a new summary of the conversation:\n\nConversation Summary:\nJames and Zeta are having an introductory conversation that has shifted from greetings to discussing conversational memory. James initially introduced himself, and Zeta offered assistance. The conversation then took a turn as James expressed interest in researching different types of conversational memory, prompting Zeta to provide an overview of the topic. Zeta explained the distinction between episodic memory (recalling specific events or conversations) and semantic memory (storing and retrieving general knowledge or facts).', additional_kwargs={}, response_metadata={})]

In [39]:
for msg in [
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    summary_pipeline_with_history.invoke(
        {"query": msg},
        config={"configurable": {"session_id": "demo2"}}
    )

In [40]:
summary_chat_map["demo2"].messages

[SystemMessage(content='Here is an updated conversation summary:\n\nConversation Summary:\nJames and Zeta continue their discussion on conversational memory, specifically buffer memory. James initially expressed interest in researching different types of conversational memory, prompting Zeta to provide an overview of the topic.\n\nZeta explained the distinction between episodic memory (recalling specific events or conversations) and semantic memory (storing and retrieving general knowledge or facts). James delved deeper into conversation buffers, specifically looking at ConversationBufferMemory and ConversationBufferWindowMemory. Zeta clarified that buffer memory is designed to manage the flow of conversation by storing a limited window of context.\n\nJames then asked for his name to be reminded, which was confirmed as "James". Zeta offered assistance with anything related to conversational memory or buffers and specifically mentioned that James had been exploring the differences betwe

In [31]:
summary_pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={"configurable": {"session_id": "demo2"}}
)

AIMessage(content="Your name is James. I'm Zeta, your helpful assistant. How can I assist you with anything related to conversational memory or buffers? You were just exploring the differences between ConversationBufferMemory and ConversationBufferWindowMemory. Would you like me to elaborate on how these concepts are used in practice?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:40:32.8879141Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25265824300, 'load_duration': 306969000, 'prompt_eval_count': 285, 'prompt_eval_duration': 15492174800, 'eval_count': 61, 'eval_duration': 9244848800, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44c1-bb43-7eb0-b94b-8e8d33eb3e58-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 285, 'output_tokens': 61, 'total_tokens': 346})

Perfect! We've successfully implemented the ConversationSummaryMemory type using the RunnableWithMessageHistory class.

**4. ConversationSummaryBufferMemory**

Our final memory type acts as a combination of ConversationSummaryMemory and ConversationBufferMemory. It keeps the buffer for the conversation up until the previous n tokens, anything beyond that limit is summarized then dropped from the buffer. Producing something like:

As with the previous memory types, we will implement this memory type again using the RunnableWithMessageHistory class. In our implementation we will modify the buffer window to be based on the number of messages rather than number of tokens. This tweak will make our implementation more closely aligned with original buffer window.

We will implement all of this via a new ConversationSummaryBufferMessageHistory class.

In [32]:
class ConversationSummaryBufferMessageHistory(BaseChatMessageHistory, BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    llm: Any
    messages: list[BaseMessage] = Field(default_factory=list)
    k: int = Field(default_factory=int)

    def __init__(self, llm: Any, k: int):
        super().__init__(llm=llm, k=k)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond"""
        existing_summary: SystemMessage | None = None
        old_messages: list[BaseMessage] | None = None
        if len(self.messages) > 0 and isinstance(self.messages[0], SystemMessage):
            print(">> Found existing summary")
            existing_summary = self.messages.pop(0)
        self.messages.extend(messages)
        if len(self.messages) > self.k:
            print(
                f">> Found {len(self.messages)} messages, dropping "
                f"oldest {len(self.messages) - self.k} messages.")
            old_messages = self.messages[:-self.k]
            self.messages = self.messages[-self.k:]
        if old_messages is None:
            print(">> No old messages to update summary with")
            return
        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensuring to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{old_messages}"
            )
        ])
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary.content if existing_summary else "",
                old_messages=[msg.content for msg in old_messages]
            )
        )
        print(f">> New summary: {new_summary.content}")
        self.messages = [SystemMessage(content=new_summary.content)] + self.messages

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []


Redefine the get_chat_history function to use our new ConversationSummaryBufferMessageHistory class.

In [33]:
summary_buffer_chat_map = {}
def get_summary_buffer_chat_history(session_id: str, k: int) -> ConversationSummaryBufferMessageHistory:
    if session_id not in summary_buffer_chat_map:
        # if session ID doesn't exist, create a new chat history
        summary_buffer_chat_map[session_id] = ConversationSummaryBufferMessageHistory(llm=llm, k=k)
    # return the chat history
    return summary_buffer_chat_map[session_id]


Setup our pipeline with new configurable fields.

In [34]:
summary_buffer_pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_summary_buffer_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)


Finally, we invoke our runnable:

In [35]:
summary_buffer_pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "demo3", "k": 4}}
)


>> No old messages to update summary with


AIMessage(content="Nice to meet you, James! I'm Zeta, your friendly assistant here to help with any questions or tasks you may have. How's your day going so far? Is there anything specific on your mind that you'd like some assistance with?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-20T09:43:36.7079587Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9931302200, 'load_duration': 584299800, 'prompt_eval_count': 30, 'prompt_eval_duration': 1432537800, 'eval_count': 51, 'eval_duration': 7762216000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e44c4-c522-7a82-9bb7-d4cc5321bba3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 51, 'total_tokens': 81})

In [36]:
history = get_summary_buffer_chat_history("demo", 4)
print(type(history))
print(history)

<class '__main__.ConversationSummaryBufferMessageHistory'>



In [37]:
print(type(summary_buffer_pipeline_with_history))
print(get_summary_buffer_chat_history)
print(get_summary_buffer_chat_history.__name__)
print(summary_chat_map)

<class 'langchain_core.runnables.history.RunnableWithMessageHistory'>
<function get_summary_buffer_chat_history at 0x0000020E38919900>
get_summary_buffer_chat_history
{'demo2': ConversationSummaryMessageHistory(messages=[SystemMessage(content='Here is the updated conversation summary:\n\nConversation Summary:\nJames and Zeta are having an introductory conversation that has shifted from greetings to discussing conversational memory. James initially introduced himself, and Zeta offered assistance. The conversation then took a turn as James expressed interest in researching different types of conversational memory, prompting Zeta to provide an overview of the topic. Zeta explained the distinction between episodic memory (recalling specific events or conversations) and semantic memory (storing and retrieving general knowledge or facts). James has since delved deeper into conversation buffers, specifically looking at ConversationBufferMemory and ConversationBufferWindowMemory.\n\nZeta clari

In [38]:
for i, msg in enumerate([
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]):
    print(f"---\nMessage {i+1}\n---\n")
    summary_buffer_pipeline_with_history.invoke(
        {"query": msg},
        config={"session_id": "demo3", "k": 4}
    )

---
Message 1
---

>> No old messages to update summary with
---
Message 2
---

>> Found 6 messages, dropping oldest 2 messages.
>> New summary: Here is a new summary of the conversation:

Conversation Summary:
James and Zeta are having an introductory conversation, where they are exchanging greetings and getting to know each other. James has introduced himself as "Hi, my name is James", and Zeta has responded with a friendly greeting, offering assistance with any questions or tasks James may have.
---
Message 3
---

>> Found existing summary
>> Found 6 messages, dropping oldest 2 messages.
>> New summary: Here is the updated conversation summary:

Conversation Summary:
James and Zeta are having an introductory conversation, where they are exchanging greetings and getting to know each other. James has introduced himself as "Hi, my name is James", and Zeta has responded with a friendly greeting, offering assistance with any questions or tasks James may have.

The conversation took a tur